In [1]:
import pandas as pd
df = pd.read_csv("customer_segments.csv")

In [2]:
df.head()

,Customer ID,recency,frequency,monetary,cluster,segment
0,12346.0,325,12,77556.46,1,Potential_Loyalist
1,12347.0,1,8,5633.32,1,Potential_Loyalist
2,12348.0,74,5,2019.40,1,Potential_Loyalist
3,12349.0,18,4,4428.69,1,Potential_Loyalist
4,12350.0,309,1,334.40,0,At_Risk


In [3]:
df["churn"] = (df["recency"] > 90).astype(int)

In [4]:
df[["recency", "churn"]].head(10)

,recency,churn
0,325,1
1,1,0
2,74,0
3,18,0
4,309,1
5,374,1
6,35,0
7,203,1
8,231,1
9,213,1


In [5]:
df["churn"].value_counts()


churn
1    2985
0    2893
Name: count, dtype: int64

In [6]:
df["churn"].value_counts(normalize=True)


churn
1    0.507826
0    0.492174
Name: proportion, dtype: float64

In [7]:
# Features (independent variables)
X = df[["recency", "frequency", "monetary"]]

# Target (dependent variable)
y = df["churn"]


In [8]:
X.shape, y.shape


((5878, 3), (5878,))

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((4408, 3), (1470, 3), (4408,), (1470,))

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [12]:
X_train_scaled.shape, X_test_scaled.shape


((4408, 3), (1470, 3))

In [13]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [14]:
y_pred = log_model.predict(X_test_scaled)


In [15]:
pd.Series(y_pred).value_counts()


0    738
1    732
Name: count, dtype: int64

In [16]:
from sklearn.metrics import confusion_matrix, classification_report

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm


array([[723,   0],
       [ 15, 732]], dtype=int64)

In [17]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       723
           1       1.00      0.98      0.99       747

    accuracy                           0.99      1470
   macro avg       0.99      0.99      0.99      1470
weighted avg       0.99      0.99      0.99      1470



In [18]:
# Predict probability of churn (class = 1)
y_prob = log_model.predict_proba(X_test_scaled)[:, 1]


In [19]:
y_prob[:10]


array([1.        , 0.00123745, 0.99992754, 0.01076375, 0.00605568,
       0.99953758, 1.        , 1.        , 1.        , 0.00919316])

In [20]:
# Create a results dataframe for test customers
results_df = X_test.copy()

results_df["actual_churn"] = y_test.values
results_df["churn_probability"] = y_prob


In [21]:
results_df["monetary"] = df.loc[X_test.index, "monetary"].values


In [22]:
results_df.head()



,recency,frequency,monetary,actual_churn,churn_probability
339,371,1,178.28,1,1.000000
4016,0,23,6849.15,0,0.001237
1125,232,11,3019.22,1,0.999928
3406,32,16,5969.93,0,0.010764
5108,24,2,514.56,0,0.006056


In [23]:
# Define high-risk threshold
risk_threshold = 0.7

# Flag high-risk customers
results_df["high_risk"] = (results_df["churn_probability"] >= risk_threshold).astype(int)


In [24]:
results_df["priority_score"] = (
    results_df["churn_probability"] * results_df["monetary"]
)


In [25]:
results_df[["churn_probability", "monetary", "priority_score"]].head()


,churn_probability,monetary,priority_score
339,1.000000,178.28,178.279999
4016,0.001237,6849.15,8.475451
1125,0.999928,3019.22,3019.001230
3406,0.010764,5969.93,64.258827
5108,0.006056,514.56,3.116008


In [26]:
def retention_action(row):
    if row["high_risk"] == 1 and row["monetary"] >= 3000:
        return "Personal Call + Premium Offer"
    elif row["high_risk"] == 1 and row["monetary"] < 3000:
        return "Discount Email / Push Notification"
    else:
        return "No Action"

results_df["retention_action"] = results_df.apply(retention_action, axis=1)


In [27]:
results_df[["churn_probability", "monetary", "priority_score", "retention_action"]].head()
results_df[["churn_probability", "monetary", "priority_score", "retention_action"]].head()


,churn_probability,monetary,priority_score,retention_action
339,1.000000,178.28,178.279999,Discount Email / Push Notification
4016,0.001237,6849.15,8.475451,No Action
1125,0.999928,3019.22,3019.001230,Personal Call + Premium Offer
3406,0.010764,5969.93,64.258827,No Action
5108,0.006056,514.56,3.116008,No Action


In [28]:
results_df.to_csv(
    "churn_retention_decisions.csv",
    index=False
)


In [30]:
import pandas as pd

sql_df = pd.read_csv("churn_retention_decisions.csv")


In [31]:
sql_df.head()


,recency,frequency,monetary,actual_churn,churn_probability,high_risk,priority_score,retention_action
0,371,1,178.28,1,1.000000,1,178.279999,Discount Email / Push Notification
1,0,23,6849.15,0,0.001237,0,8.475451,No Action
2,232,11,3019.22,1,0.999928,1,3019.001230,Personal Call + Premium Offer
3,32,16,5969.93,0,0.010764,0,64.258827,No Action
4,24,2,514.56,0,0.006056,0,3.116008,No Action
